# Mouse Trajectory Synthesis - Visualizations

Generates three key figures for the README.  
**Prerequisite:** run `python setup_data.py` from the repo root so that `data/` contains checkpoints and the demo pool.

In [ ]:
"""Cell 1: Imports and setup."""
import math
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

# Ensure repo root is on sys.path so we can import project modules.
# When running from notebooks/, the repo root is one level up.
# When running from repo root, Path.cwd() is already correct.
_cwd = Path.cwd()
REPO_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from features import FEATURE_NAMES, extract_feature_matrix

DATA_DIR = REPO_ROOT / "data"
FIG_DIR = REPO_ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

# Consistent, clean style
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 200,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print(f"Repo root: {REPO_ROOT}")
print(f"Data dir:  {DATA_DIR}")
print(f"Figures:   {FIG_DIR}")

In [ ]:
"""Cell 2: Load demo pool and initialize generators."""

# --- Load human demo pool ---
pool = np.load(DATA_DIR / "demo_pool.npz")
pool_flat = pool["flat"]       # (N_total, 2) float32 - x, y positions
pool_offsets = pool["offsets"]  # (M+1,) int64 - start/end indices per trajectory
pool_t = pool["t"]             # (N_total,) float32 - timestamps (relative)
pool_meta = pool["meta"]       # (M, 3) float32 - log_dist, cos_angle, sin_angle
n_human = len(pool_meta)
print(f"Demo pool: {n_human} human trajectories")


def get_human_trajectory(idx):
    """Extract trajectory idx from the demo pool as List[(x, y, t)]."""
    lo, hi = int(pool_offsets[idx]), int(pool_offsets[idx + 1])
    xy = pool_flat[lo:hi]
    t = pool_t[lo:hi]
    return [(float(xy[i, 0]), float(xy[i, 1]), float(t[i])) for i in range(hi - lo)]


# --- Initialize DDPM generator ---
from experiments.ddpm_arclen import generate_path as ddpm_generate_path

# --- Initialize VQ-VAE + Transformer generator ---
from experiments.vqvae_ar_transformer import generate_path as vqvae_generate_path

print("Generators loaded.")

In [ ]:
"""Cell 3: Figure 1 - Trajectory Overlay.

Pick a human trajectory, extract its start/end, generate DDPM and VQ-VAE
trajectories for the same endpoints, and plot all three.
"""
# Pick a human trajectory with moderate distance for visual clarity
rng = np.random.default_rng(7)
candidates = []
for i in range(n_human):
    log_d = float(pool_meta[i, 0])
    dist = math.exp(log_d)
    if 200 < dist < 600:  # moderate-distance movements
        candidates.append(i)

chosen_idx = rng.choice(candidates)
human_traj = get_human_trajectory(chosen_idx)
human_pts = np.array(human_traj)

sx, sy = human_pts[0, 0], human_pts[0, 1]
ex, ey = human_pts[-1, 0], human_pts[-1, 1]
print(f"Human trajectory {chosen_idx}: ({sx:.0f}, {sy:.0f}) -> ({ex:.0f}, {ey:.0f}), "
      f"dist={math.hypot(ex - sx, ey - sy):.0f} px, {len(human_traj)} points")

# Generate synthetic trajectories for the same start/end
ddpm_traj = ddpm_generate_path(sx, sy, ex, ey)
vqvae_traj = vqvae_generate_path(sx, sy, ex, ey)

ddpm_pts = np.array(ddpm_traj)
vqvae_pts = np.array(vqvae_traj)

# Plot
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(human_pts[:, 0], human_pts[:, 1],
        color="#2176FF", linewidth=1.8, alpha=0.9, label="Human", zorder=3)
ax.plot(ddpm_pts[:, 0], ddpm_pts[:, 1],
        color="#F77F00", linewidth=1.5, alpha=0.85, label="DDPM", zorder=2)
ax.plot(vqvae_pts[:, 0], vqvae_pts[:, 1],
        color="#06D6A0", linewidth=1.5, alpha=0.85, label="VQ-VAE + Transformer", zorder=2)

# Mark start and end
ax.scatter([sx], [sy], color="black", s=50, zorder=5, marker="o", label="Start")
ax.scatter([ex], [ey], color="black", s=50, zorder=5, marker="x", label="End")

ax.set_title("Real vs. Generated Trajectories", fontsize=13, fontweight="bold")
ax.set_xlabel("x (px)", fontsize=10, color="#555")
ax.set_ylabel("y (px)", fontsize=10, color="#555")
ax.legend(loc="best", framealpha=0.9, fontsize=9)
ax.set_aspect("equal")
ax.grid(False)

fig.tight_layout()
fig.savefig(FIG_DIR / "trajectory_overlay.png", bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_DIR / 'trajectory_overlay.png'}")

In [ ]:
"""Cell 4: Figure 2 - Feature Distribution Comparison.

Extract features from ~500 human, ~500 DDPM, and ~500 VQ-VAE trajectories.
Violin plots for 5 key features side-by-side.
"""
N_SAMPLE = 500
rng = np.random.default_rng(42)

# --- Sample human trajectories ---
human_indices = rng.choice(n_human, size=min(N_SAMPLE, n_human), replace=False)
human_trajs = [get_human_trajectory(int(i)) for i in human_indices]
human_features = extract_feature_matrix(human_trajs)
print(f"Human features: {human_features.shape}")

# --- Generate DDPM trajectories with matched distances ---
ddpm_trajs = []
for idx in human_indices:
    traj = get_human_trajectory(int(idx))
    pts = np.array(traj)
    s_x, s_y = pts[0, 0], pts[0, 1]
    e_x, e_y = pts[-1, 0], pts[-1, 1]
    ddpm_trajs.append(ddpm_generate_path(s_x, s_y, e_x, e_y))

ddpm_features = extract_feature_matrix(ddpm_trajs)
print(f"DDPM features:  {ddpm_features.shape}")

# --- Generate VQ-VAE trajectories with matched distances ---
vqvae_trajs = []
for idx in human_indices:
    traj = get_human_trajectory(int(idx))
    pts = np.array(traj)
    s_x, s_y = pts[0, 0], pts[0, 1]
    e_x, e_y = pts[-1, 0], pts[-1, 1]
    vqvae_trajs.append(vqvae_generate_path(s_x, s_y, e_x, e_y))

vqvae_features = extract_feature_matrix(vqvae_trajs)
print(f"VQ-VAE features: {vqvae_features.shape}")

# --- Select 5 key features ---
key_features = [
    "mean_velocity",
    "curvature_mean",
    "angular_velocity_mean",
    "path_efficiency",
    "num_direction_changes",
]
key_indices = [FEATURE_NAMES.index(f) for f in key_features]

# --- Violin plots ---
fig, axes = plt.subplots(1, 5, figsize=(16, 4.5))

colors = {
    "Human": "#2176FF",
    "DDPM": "#F77F00",
    "VQ-VAE": "#06D6A0",
}

for ax, feat_name, feat_idx in zip(axes, key_features, key_indices):
    h_vals = human_features[:, feat_idx]
    d_vals = ddpm_features[:, feat_idx]
    v_vals = vqvae_features[:, feat_idx]

    # Clip extreme outliers for readability (keep 1st-99th percentile range)
    all_vals = np.concatenate([h_vals, d_vals, v_vals])
    lo, hi = np.percentile(all_vals, [1, 99])
    margin = (hi - lo) * 0.1
    clip_lo, clip_hi = lo - margin, hi + margin

    data = [
        np.clip(h_vals, clip_lo, clip_hi),
        np.clip(d_vals, clip_lo, clip_hi),
        np.clip(v_vals, clip_lo, clip_hi),
    ]

    parts = ax.violinplot(data, positions=[1, 2, 3], showmedians=True, showextrema=False)

    for i, (body, label) in enumerate(zip(parts["bodies"], colors)):
        body.set_facecolor(colors[label])
        body.set_alpha(0.7)
    parts["cmedians"].set_color("black")
    parts["cmedians"].set_linewidth(1.5)

    # Clean up feature name for display
    display_name = feat_name.replace("_", " ").title()
    ax.set_title(display_name, fontsize=9, fontweight="bold")
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(["Human", "DDPM", "VQ-VAE"], fontsize=8, rotation=15)
    ax.grid(False)

fig.suptitle("Feature Distribution Comparison", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "feature_distributions.png", bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_DIR / 'feature_distributions.png'}")

In [ ]:
"""Cell 5: Figure 3 - AUC Progression Chart.

Bar chart showing AUC by architecture family. Lower = more human-like.
"""

approaches = [
    ("Parametric\n(sigma-lognormal)",  0.998, "generative"),
    ("CFM",                            0.919, "generative"),
    ("Stall injection",                0.93,  "generative"),
    ("VQ-VAE +\nTransformer",          0.89,  "generative"),
    ("DDPM",                           0.862, "generative"),
    ("Perturbed\nreplay",              0.70,  "replay"),
    ("Corpus replay",                  0.50,  "replay"),
]

labels = [a[0] for a in approaches]
aucs = [a[1] for a in approaches]
types = [a[2] for a in approaches]

color_gen = "#5E60CE"   # purple for generative
color_rep = "#64DFDF"   # teal for replay-based
bar_colors = [color_gen if t == "generative" else color_rep for t in types]

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(range(len(labels)), aucs, color=bar_colors, edgecolor="white", width=0.7)

# Value labels on bars
for bar, auc in zip(bars, aucs):
    y_pos = bar.get_height() + 0.012
    ax.text(bar.get_x() + bar.get_width() / 2, y_pos,
            f"{auc:.3f}" if auc not in (0.93, 0.50) else f"~{auc:.2f}",
            ha="center", va="bottom", fontsize=9, fontweight="bold")

# Theoretical minimum line
ax.axhline(y=0.50, color="#E63946", linestyle="--", linewidth=1.5, alpha=0.8,
           label="Theoretical minimum (0.50)")

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("AUC (lower = more human-like)", fontsize=11)
ax.set_title("AUC by Architecture Family (lower = more human-like)",
             fontsize=13, fontweight="bold")
ax.set_ylim(0.0, 1.15)
ax.grid(False)

# Legend for color coding
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=color_gen, label="Generative"),
    Patch(facecolor=color_rep, label="Replay-based"),
    plt.Line2D([0], [0], color="#E63946", linestyle="--", linewidth=1.5, label="Theoretical min (0.50)"),
]
ax.legend(handles=legend_elements, loc="upper right", framealpha=0.9, fontsize=9)

fig.tight_layout()
fig.savefig(FIG_DIR / "auc_progression.png", bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_DIR / 'auc_progression.png'}")